# Data on the Line · Kurulum testi (5 dakika)

Bu notebook, oturumda kullanacağınız çalışma ortamının hazır olup olmadığını kontrol eder. Vakada da aynı adımları izleyeceksiniz.

1. Notebook'u size verilen bağlantıyla ya da QR kodla Colab'de açın. Veri dosyaları GitHub'dan otomatik olarak indirilir.
2. *Çalışma zamanı → Tümünü çalıştır* ile bütün hücreleri sırayla çalıştırın. Sonunda **“Her şey hazır.”** yazısını görmelisiniz.

Kendi bilgisayarınızda Jupyter kullanmak isterseniz `Kurulum_Testi` klasörünü indirip notebook'u orada açabilirsiniz.

Bir adım hata verirse çıkan mesajı oturumdan önce eğitmeninize gönderin.

In [ ]:
# Colab'de veri dosyaları GitHub'dan otomatik olarak indirilir; kendi bilgisayarınızda bu adım atlanır
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("data"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/burcu-koca-kd/data-on-the-line.git", "/content/data-on-the-line"], check=True)
    os.chdir("/content/data-on-the-line/tr/1_Hazırlık/Kurulum_Testi")
from pathlib import Path
print("Çalışma klasörü:", Path.cwd())
print("data klasörü bulundu ✓" if Path("data").exists() else "data klasörü BULUNAMADI: klasör yolunu kontrol edin.")

In [ ]:
def need(module, package=None):
    """Kütüphaneyi içe aktarır; kurulu değilse önce kurar."""
    try:
        return __import__(module)
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q " + (package or module))
        return __import__(module)

need("highspy")                                            # yedek çözücü: bazı Mac'lerde gerekiyor
from importlib.metadata import version
try:
    pulp_major = int(version("pulp").split(".")[0])
except Exception:
    pulp_major = 0
if not 2 <= pulp_major < 4:                                # vaka pulp 3 yazımını kullanıyor; Colab'de hazır gelen pulp 4'te bu yazım değişti
    get_ipython().run_line_magic("pip", 'install -q "pulp>=2.7,<4"')
    for m in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[m]
import pulp
statsmodels, sklearn = need("statsmodels"), need("sklearn", "scikit-learn")
import pandas as pd, numpy as np, zipfile
print("pulp", pulp.__version__, "✓ | pandas", pd.__version__, "✓ | statsmodels", statsmodels.__version__, "✓ | scikit-learn", sklearn.__version__, "✓")

In [ ]:
result = {}
result["Veri dosyası okunuyor"] = len(pd.read_csv("data/sample.csv")) == 3

with zipfile.ZipFile("release_test.zip") as z:            # 45. ve 95. dakikalarda tam olarak bunu yapacaksınız
    z.extractall(".", pwd=b"test")
result["Şifreli dosya açılıyor"] = Path("release/test.csv").exists()

def solves(solver):
    """Verilen çözücü küçük bir modeli çözebiliyor mu? Çözebiliyorsa True döndürür."""
    try:
        m = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable.dicts("x", range(3), cat="Binary")
        m += pulp.lpSum(x.values()); m += x[0] + x[1] >= 1; m += x[1] + x[2] >= 1
        m.solve(solver)
        return pulp.LpStatus[m.status] == "Optimal" and pulp.value(m.objective) == 1
    except Exception:
        return False

solver = "CBC" if solves(pulp.PULP_CBC_CMD(msg=False)) else "HiGHS" if solves(pulp.HiGHS(msg=False)) else None
result["Küçük bir tamsayılı model çözülüyor" + (f" ({solver})" if solver else "")] = solver is not None

for name, ok in result.items():
    print(("✓ " if ok else "✗ ") + name)
if not solver:
    print("\nÇözücü başlatılamadı. En kolay çözüm Colab'i kullanmak. Kendi Mac'inizde çalışacaksanız Terminal'de `pip install highspy` komutunu çalıştırın, ardından çekirdeği yeniden başlatıp bu notebook'u baştan çalıştırın.")
print("\nHer şey hazır." if all(result.values()) else "\nBir adım başarısız oldu. Lütfen bu mesajı eğitmeninize gönderin.")